In [4]:
USE_GPU = True   # Change to False if you want CPU only

import torch
GPU_AVAILABLE = torch.cuda.is_available() and USE_GPU
print(f"GPU mode: {'Enabled' if GPU_AVAILABLE else 'Disabled'}")

ImportError: DLL load failed while importing _C: The specified procedure could not be found.

In [5]:

import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage import io, exposure, color, feature
from skimage.restoration import denoise_tv_chambolle
from skimage.feature import local_binary_pattern
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier

from catboost import CatBoostClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import pickle
import time
import seaborn as sns

%matplotlib inline

class Timer:
    def __init__(self, name): self.name = name
    def __enter__(self): self.start = time.time(); return self
    def __exit__(self, *args): print(f"{self.name} Done — {time.time()-self.start:.2f}s")

In [6]:
TRAIN_IMS_DIR = Path("data/train_ims")
TEST_IMS_DIR = Path("data/test_ims")
TRAIN_CSV = Path("data/train.csv")
TEST_CSV = Path("data/test.csv")

FEATURES_DIR = Path("features/")
MODEL_DIR = Path("models/")

for d in [FEATURES_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

In [7]:
def extract_rich_features(image_path):
    img = io.imread(image_path)
    if len(img.shape) != 3:
        img = np.stack([img]*3, axis=-1)
    
    gray = color.rgb2gray(img)
    
    # Light preprocessing only
    gray = exposure.equalize_adapthist(gray, clip_limit=0.02)
    gray = denoise_tv_chambolle(gray, weight=0.03)
    
    features = []
    
    # 1. Multi-scale HOG (better than single scale)
    for cell_size in [(4,4), (8,8)]:
        hog_feat = feature.hog(gray, orientations=9, pixels_per_cell=cell_size,
                               cells_per_block=(2,2), block_norm='L2')
        features.extend(hog_feat)
    
    # 2. Multi-scale LBP (very effective on 32x32)
    for radius, points in [(1,8), (2,16), (3,24)]:
        lbp = local_binary_pattern(gray, points, radius, method='uniform')
        hist, _ = np.histogram(lbp.ravel(), bins=points + 2, density=True)
        features.extend(hist)
    
    # 3. Strong color features from ORIGINAL RGB
    for c in range(3):
        ch = img[:,:,c].astype(float)
        features.extend([
            np.mean(ch), np.std(ch), np.max(ch),
            np.percentile(ch, 25), np.percentile(ch, 75)
        ])
    
    # 4. Simple flip augmentation (average)
    flipped_gray = np.fliplr(gray)
    hog_flip = feature.hog(flipped_gray, orientations=9, pixels_per_cell=(4,4),
                           cells_per_block=(2,2), block_norm='L2')
    features = (np.array(features) + np.concatenate([hog_flip, features[len(hog_flip):]])) / 2.0
    
    return np.array(features)

In [8]:
def extract_all_features(df, ims_dir, save_name):
    feature_file = FEATURES_DIR / save_name
    if feature_file.exists():
        print(f"Loading cached {save_name}")
        with open(feature_file, 'rb') as f:
            return pickle.load(f)
    
    print(f"Extracting features for {save_name}...")
    feats, labs = [], []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        path = ims_dir / row['im_name']
        feat = extract_rich_features(str(path))
        feats.append(feat)
        labs.append(row['label'])
    
    feats = np.array(feats)
    labs = np.array(labs)
    with open(feature_file, 'wb') as f:
        pickle.dump((feats, labs), f)
    return feats, labs

with Timer("Feature Extraction"):
    train_df = pd.read_csv(TRAIN_CSV)
    test_df = pd.read_csv(TEST_CSV)
    
    train_features, train_labels = extract_all_features(train_df, TRAIN_IMS_DIR, "train_rich_v2.pkl")
    test_features, _ = extract_all_features(test_df, TEST_IMS_DIR, "test_rich_v2.pkl")

Extracting features for train_rich_v2.pkl...


  0%|          | 0/50000 [00:00<?, ?it/s]c:\Users\cake3\.conda\envs\cuml_env\lib\site-packages\skimage\feature\texture.py:360: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
100%|██████████| 50000/50000 [06:10<00:00, 134.79it/s]


Extracting features for test_rich_v2.pkl...


100%|██████████| 10000/10000 [01:40<00:00, 99.15it/s]


Feature Extraction Done — 473.38s


In [ ]:
with Timer("Scaling & Training"):
    scaler = StandardScaler()
    train_features = scaler.fit_transform(train_features)
    test_features = scaler.transform(test_features)
    
    rf = RandomForestClassifier(n_estimators=800, n_jobs=-1, random_state=42, max_depth=18, max_features='log2')
    et = ExtraTreesClassifier(n_estimators=600, n_jobs=-1, random_state=42)
    cb = CatBoostClassifier(
        iterations=1200,
        depth=8,
        learning_rate=0.07,
        verbose=False,
        random_seed=42,
        task_type='GPU'
    )
    model = VotingClassifier(
        estimators=[('rf', rf), ('et', et), ('cb', cb)],
        voting='soft',
        n_jobs=-1
    )
    
    model.fit(train_features, train_labels)
    
    joblib.dump(model, MODEL_DIR / "ensemble_v3.pkl")
    joblib.dump(scaler, MODEL_DIR / "scaler_v3.pkl")
    print("Model saved!")

Scaling & Training Done — 1.14s


NameError: name 'GPU_AVAILABLE' is not defined

In [ ]:
with Timer("Prediction"):
    test_pred = model.predict(test_features)
    test_df['label'] = test_pred
    test_df[['im_name', 'label']].to_csv("submission_v3.csv", index=False)
    print("Submission saved as submission_v3.csv")
    
    # Quick validation confusion matrix
    val_size = 5000
    val_pred = model.predict(train_features[:val_size])
    cm = confusion_matrix(train_labels[:val_size], val_pred)
    plt.figure(figsize=(10,8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Validation Subset)')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()